In [7]:
import cv2
import numpy as np
from keras.models import load_model
from ultralytics import YOLO
from collections import deque

# 加载模型
classification_model = load_model('./models/classification.keras')
detection_model = YOLO("./models/detection.pt")
detection_model.overrides['conf'] = 0.5  # 置信度阈值
detection_model.overrides['iou'] = 0.5   # IoU 阈值

# 食物类别
classification_labels = ["Apple pie", "Baby back ribs", "Baklava", "Beef carpaccio", "Beef tartare", "Beet salad", "Beignets", "Bibimbap",
    "Bread pudding", "Breakfast burrito", "Bruschetta", "Caesar salad", "Cannoli", "Caprese salad", "Carrot cake",
    "Ceviche", "Cheesecake", "Cheese plate", "Chicken curry", "Chicken quesadilla", "Chicken wings", "Chocolate cake",
    "Chocolate mousse", "Churros", "Clam chowder", "Club sandwich", "Crab cakes", "Creme brulee", "Croque madame",
    "Cup cakes", "Deviled eggs", "Donuts", "Dumplings", "Edamame", "Eggs benedict", "Escargots", "Falafel",
    "Filet mignon", "Fish and chips", "Foie gras", "French fries", "French onion soup", "French toast", "Fried calamari",
    "Fried rice", "Frozen yogurt", "Garlic bread", "Gnocchi", "Greek salad", "Grilled cheese sandwich", "Grilled salmon",
    "Guacamole", "Gyoza", "Hamburger", "Hot and sour soup", "Hot dog", "Huevos rancheros", "Hummus", "Ice cream",
    "Lasagna", "Lobster bisque", "Lobster roll sandwich", "Macaroni and cheese", "Macarons", "Miso soup", "Mussels",
    "Nachos", "Omelette", "Onion rings", "Oysters", "Pad thai", "Paella", "Pancakes", "Panna cotta", "Peking duck",
    "Pho", "Pizza", "Pork chop", "Poutine", "Prime rib", "Pulled pork sandwich", "Ramen", "Ravioli", "Red velvet cake",
    "Risotto", "Samosa", "Sashimi", "Scallops", "Seaweed salad", "Shrimp and grits", "Spaghetti bolognese",
    "Spaghetti carbonara", "Spring rolls", "Steak", "Strawberry shortcake", "Sushi", "Tacos", "Takoyaki", "Tiramisu",
    "Tuna tar tare", "Waffles"]

# 记录状态
detecting_food = False  
recorded = False  

# 维护检测结果缓冲区
category_buffers = {}  # 存储每个食物框的分类结果
BUFFER_SIZE = 5  # 只存储最近5帧的分类结果
STABLE_THRESHOLD = 3  # 至少有3帧是同一类别才更新

def preprocess_image(img, target_size):
    """预处理分类模型输入"""
    img_resized = cv2.resize(img, (target_size, target_size))
    img_normalized = img_resized / 255.0
    img_normalized = (img_normalized - 0.5) / 0.5
    return np.expand_dims(img_normalized, axis=0)

def iou(boxA, boxB):
    """计算两个矩形框的交并比 (IoU)"""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou

def process_frame(frame):
    """检测食物并分类"""
    global detecting_food, recorded, category_buffers

    results = detection_model(frame, stream=True)
    current_detecting_food = False  
    detected_items = []  

    new_buffers = {}  # 记录本帧的检测框，以便和上一帧匹配

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf.item()
            if confidence < 0.5:
                continue

            current_detecting_food = True
            cropped_img = frame[y1:y2, x1:x2]
            if cropped_img.size == 0:
                continue

            preprocessed_img = preprocess_image(cropped_img, 640)
            predictions = classification_model.predict(preprocessed_img, verbose=0)
            predicted_class = np.argmax(predictions[0])
            confidence_cls = predictions[0][predicted_class]
            category = classification_labels[predicted_class]

            # 计算该框和之前帧的检测框的 IoU
            best_match_id = None
            best_iou = 0
            for prev_id, (prev_box, prev_buffer) in category_buffers.items():
                iou_score = iou((x1, y1, x2, y2), prev_box)
                if iou_score > best_iou and iou_score > 0.5:  # IoU 大于 0.5 认为是同一个物体
                    best_iou = iou_score
                    best_match_id = prev_id

            if best_match_id is not None:
                # 找到匹配的旧框，使用其缓冲区
                buffer = category_buffers[best_match_id][1]
            else:
                # 新检测框，创建缓冲区
                buffer = deque(maxlen=BUFFER_SIZE)

            # 添加新分类结果
            buffer.append(category)

            # 计算出现次数最多的类别
            most_common_category = max(set(buffer), key=buffer.count)
            detected_items.append(most_common_category)

            # 保存当前检测框
            new_buffers[id(buffer)] = ((x1, y1, x2, y2), buffer)

            # 显示稳定后的分类结果
            label = f"{most_common_category}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # 更新缓存
    category_buffers = new_buffers

    # 逻辑处理
    if detecting_food and not current_detecting_food:
        recorded = False
        print("食物已移除，等待新一轮检测...")

    if current_detecting_food and not recorded:
        print("检测到新的食物组合:", detected_items)
        recorded = True

    detecting_food = current_detecting_food

    return frame

def main():
    """打开摄像头并进行实时检测"""
    cap = cv2.VideoCapture(1)

    if not cap.isOpened():
        cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to capture frame.")
            break

        processed_frame = process_frame(frame)
        cv2.imshow("Food Detection and Classification", processed_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


0: 480x640 (no detections), 314.3ms
Speed: 3.0ms preprocess, 314.3ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 food, 236.2ms
Speed: 5.5ms preprocess, 236.2ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 food, 223.2ms
Speed: 3.1ms preprocess, 223.2ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 food, 213.4ms
Speed: 6.3ms preprocess, 213.4ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 food, 252.6ms
Speed: 6.0ms preprocess, 252.6ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 food, 187.1ms
Speed: 4.5ms preprocess, 187.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 food, 270.2ms
Speed: 4.1ms preprocess, 270.2ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 253.8ms
Speed: 3.7ms preprocess, 253.8ms inference, 0.0ms postprocess per image 

In [1]:
import cv2
import numpy as np
from keras.models import load_model
from ultralytics import YOLO

# Load the classification model
classification_model = load_model('./models/classification.keras')

# Load YOLO detection model
detection_model = YOLO("./models/detection.pt")

# List of classification labels (adjust as needed)
classification_labels = [
    "Apple pie", "Baby back ribs", "Baklava", "Beef carpaccio", "Beef tartare", "Beet salad", "Beignets", "Bibimbap",
    "Bread pudding", "Breakfast burrito", "Bruschetta", "Caesar salad", "Cannoli", "Caprese salad", "Carrot cake",
    "Ceviche", "Cheesecake", "Cheese plate", "Chicken curry", "Chicken quesadilla", "Chicken wings", "Chocolate cake",
    "Chocolate mousse", "Churros", "Clam chowder", "Club sandwich", "Crab cakes", "Creme brulee", "Croque madame",
    "Cup cakes", "Deviled eggs", "Donuts", "Dumplings", "Edamame", "Eggs benedict", "Escargots", "Falafel",
    "Filet mignon", "Fish and chips", "Foie gras", "French fries", "French onion soup", "French toast", "Fried calamari",
    "Fried rice", "Frozen yogurt", "Garlic bread", "Gnocchi", "Greek salad", "Grilled cheese sandwich", "Grilled salmon",
    "Guacamole", "Gyoza", "Hamburger", "Hot and sour soup", "Hot dog", "Huevos rancheros", "Hummus", "Ice cream",
    "Lasagna", "Lobster bisque", "Lobster roll sandwich", "Macaroni and cheese", "Macarons", "Miso soup", "Mussels",
    "Nachos", "Omelette", "Onion rings", "Oysters", "Pad thai", "Paella", "Pancakes", "Panna cotta", "Peking duck",
    "Pho", "Pizza", "Pork chop", "Poutine", "Prime rib", "Pulled pork sandwich", "Ramen", "Ravioli", "Red velvet cake",
    "Risotto", "Samosa", "Sashimi", "Scallops", "Seaweed salad", "Shrimp and grits", "Spaghetti bolognese",
    "Spaghetti carbonara", "Spring rolls", "Steak", "Strawberry shortcake", "Sushi", "Tacos", "Takoyaki", "Tiramisu",
    "Tuna tartare", "Waffles"
]

def resize_with_padding(img, target_size):
    """Resize image with padding to maintain aspect ratio."""
    old_size = img.shape[:2]  # Original size (height, width)
    ratio = float(target_size) / max(old_size)  # Scaling ratio
    new_size = tuple([int(x * ratio) for x in old_size])  # New size after scaling

    # Resize image
    resized_img = cv2.resize(img, (new_size[1], new_size[0]), interpolation=cv2.INTER_AREA)

    # Calculate padding
    delta_w = target_size - new_size[1]
    delta_h = target_size - new_size[0]
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)

    # Add padding (black background)
    color = [0, 0, 0]
    new_img = cv2.copyMakeBorder(resized_img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    return new_img

def process_image(image_path):
    """Detect food with YOLO and classify each detected food."""
    # Load the input image
    frame = cv2.imread(image_path)
    if frame is None:
        print("Error: Could not read image.")
        return

    # Detect food using YOLO
    results = detection_model(frame)

    for result in results:
        for box in result.boxes:
            # Extract bounding box coordinates
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf.item()
            class_id = int(box.cls)

            # Crop the detected food region
            cropped_img = frame[y1:y2, x1:x2]

            # Resize and preprocess for classification model
            resized_img = resize_with_padding(cropped_img, 640)
            img_normalized = resized_img / 255.0  # Normalize image
            img_expanded = np.expand_dims(img_normalized, axis=0)  # Add batch dimension

            # Predict food category
            predictions = classification_model.predict(img_expanded)
            predicted_class = np.argmax(predictions[0])
            confidence_cls = predictions[0][predicted_class]

            # Display detection results
            label = f"{classification_labels[predicted_class]} ({confidence_cls:.2f})"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Show the final image with detections and classifications
    cv2.imshow("Food Detection and Classification", frame)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    # Specify the path to your input image
    image_path = "./images/2.jpg"
    process_image(image_path)

D:\conda\envs\opencv\lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 33 variables whereas the saved optimizer has 64 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



0: 480x640 1 food, 255.8ms
Speed: 6.0ms preprocess, 255.8ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step


In [11]:
import cv2
import numpy as np
import os
import time
from keras.models import load_model
from ultralytics import YOLO
from collections import deque

# 加载模型
classification_model = load_model('./models/classification.keras')
detection_model = YOLO("./models/detection.pt")
detection_model.overrides['conf'] = 0.5  # 置信度阈值
detection_model.overrides['iou'] = 0.5   # IoU 阈值

# 目标跟踪
BUFFER_SIZE = 5  # 需要累计 5 帧才能分类
IOU_THRESHOLD = 0.8  # 用于判断检测框是否稳定
trackers = {}  # 目标ID -> (坐标, 帧计数, 已稳定, 累积帧, 失效帧)

# 输出目录
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 食物类别
classification_labels = ["Apple pie", "Baby back ribs", "Baklava", "Beef carpaccio", "Beef tartare", "Beet salad", "Beignets", "Bibimbap",
    "Bread pudding", "Breakfast burrito", "Bruschetta", "Caesar salad", "Cannoli", "Caprese salad", "Carrot cake",
    "Ceviche", "Cheesecake", "Cheese plate", "Chicken curry", "Chicken quesadilla", "Chicken wings", "Chocolate cake",
    "Chocolate mousse", "Churros", "Clam chowder", "Club sandwich", "Crab cakes", "Creme brulee", "Croque madame",
    "Cup cakes", "Deviled eggs", "Donuts", "Dumplings", "Edamame", "Eggs benedict", "Escargots", "Falafel",
    "Filet mignon", "Fish and chips", "Foie gras", "French fries", "French onion soup", "French toast", "Fried calamari",
    "Fried rice", "Frozen yogurt", "Garlic bread", "Gnocchi", "Greek salad", "Grilled cheese sandwich", "Grilled salmon",
    "Guacamole", "Gyoza", "Hamburger", "Hot and sour soup", "Hot dog", "Huevos rancheros", "Hummus", "Ice cream",
    "Lasagna", "Lobster bisque", "Lobster roll sandwich", "Macaroni and cheese", "Macarons", "Miso soup", "Mussels",
    "Nachos", "Omelette", "Onion rings", "Oysters", "Pad thai", "Paella", "Pancakes", "Panna cotta", "Peking duck",
    "Pho", "Pizza", "Pork chop", "Poutine", "Prime rib", "Pulled pork sandwich", "Ramen", "Ravioli", "Red velvet cake",
    "Risotto", "Samosa", "Sashimi", "Scallops", "Seaweed salad", "Shrimp and grits", "Spaghetti bolognese",
    "Spaghetti carbonara", "Spring rolls", "Steak", "Strawberry shortcake", "Sushi", "Tacos", "Takoyaki", "Tiramisu",
    "Tuna tartare", "Waffles"]

def preprocess_image(img, target_size=640):
    """预处理分类模型输入"""
    img_resized = cv2.resize(img, (target_size, target_size))
    img_normalized = img_resized / 255.0
    img_normalized = (img_normalized - 0.5) / 0.5
    return np.expand_dims(img_normalized, axis=0)

def iou(boxA, boxB):
    """计算两个矩形框的交并比 (IoU)"""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    return interArea / float(boxAArea + boxBArea - interArea)

def process_frame(frame):
    """检测食物并分类"""
    global trackers

    results = detection_model(frame, stream=True)
    detected_objects = {}  # 记录新一帧的检测框

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            confidence = box.conf.item()
            if confidence < 0.5:
                continue

            # 目标匹配
            best_match_id = None
            best_iou = 0
            for obj_id, (prev_box, count, stable, buffer, lost_frames) in trackers.items():
                iou_score = iou((x1, y1, x2, y2), prev_box)
                if iou_score > best_iou and iou_score > IOU_THRESHOLD:
                    best_iou = iou_score
                    best_match_id = obj_id

            if best_match_id is not None:
                count, stable, buffer, lost_frames = trackers[best_match_id][1:]
                count += 1
                lost_frames = 0
                if count >= BUFFER_SIZE:
                    stable = True  # 达到 5 帧，标记为稳定
            else:
                best_match_id = len(trackers) + 1
                count, stable, buffer, lost_frames = 1, False, deque(maxlen=BUFFER_SIZE), 0

            detected_objects[best_match_id] = ((x1, y1, x2, y2), count, stable, buffer, lost_frames)

            final_category = "Unknown"  # 确保变量有默认值
            if stable and len(buffer) < BUFFER_SIZE:
                cropped_img = frame[y1:y2, x1:x2]
                if cropped_img.size == 0:
                    continue
                preprocessed_img = preprocess_image(cropped_img)
                predictions = classification_model.predict(preprocessed_img, verbose=0)
                predicted_class = np.argmax(predictions[0])
                category = classification_labels[predicted_class]
                buffer.append(category)

                if len(buffer) == BUFFER_SIZE:
                    final_category = max(set(buffer), key=buffer.count)
                    timestamp = int(time.time())
                    filename = f"{OUTPUT_DIR}/{final_category}_{timestamp}.jpg"
                    cv2.imwrite(filename, cropped_img)
                    print(f"保存：{filename}")

            # 画框
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, final_category, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    trackers = detected_objects
    return frame

def main():
    cap = cv2.VideoCapture(2)

    if not cap.isOpened():
        cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return
        
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        processed_frame = process_frame(frame)
        cv2.imshow("Food Detection", processed_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


0: 384x640 2 foods, 243.0ms
Speed: 4.0ms preprocess, 243.0ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 foods, 258.1ms
Speed: 3.0ms preprocess, 258.1ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 foods, 241.1ms
Speed: 3.6ms preprocess, 241.1ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 foods, 229.8ms
Speed: 4.0ms preprocess, 229.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 foods, 263.6ms
Speed: 3.0ms preprocess, 263.6ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 foods, 258.8ms
Speed: 4.0ms preprocess, 258.8ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 foods, 250.7ms
Speed: 4.2ms preprocess, 250.7ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 foods, 237.4ms
Speed: 6.0ms preprocess, 237.4ms inference, 0.0ms postprocess per image at shape (

In [13]:
import requests

print("111")

111
